# Notebook 3 - Metro Madrid: pipeline híbrido final para forecasting diario a 30 días

Este notebook sustituye la lógica de comparativa de modelos del notebook 2 por **un único pipeline final**, diseñado específicamente para la demanda diaria de Metro Madrid. La idea central es separar con más rigor la parte explicable por calendario y régimen, para dejar a la segunda etapa únicamente la dinámica residual de corto plazo.

El resultado es un enfoque más apropiado para esta serie porque:
- modela explícitamente festivos reales, puentes, Semana Santa, estructura semanal y estacionalidad anual suave,
- trata la ruptura COVID/post-COVID como un cambio estructural y no como un simple outlier,
- reserva el modelo flexible para el residuo, donde realmente aporta valor,
- y evalúa el sistema completo con **rolling-origin backtesting serio sobre 2024** y con métricas puntuales y probabilísticas.

## Diseño metodológico

**Etapa 1: bloque determinista en `log1p(y)`**

Usamos un modelo lineal regularizado para maximizar interpretabilidad y controlar bien el calendario: día de la semana, laborable/fin de semana, mes, semana del año, inicio y fin de mes, proximidad al fin de mes, festivos nacionales y de la Comunidad de Madrid, vísperas, post-festivos, puentes, ventana de Semana Santa, Fourier anual y variables de intervención para COVID.

**Etapa 2: bloque residual directo multi-horizonte**

Sobre el residuo de la etapa 1 entrenamos un modelo de boosting temporal directo a 30 horizontes con `lags`, medias móviles, desviaciones móviles y diferencias respecto a la semana previa. Se entrena como un único modelo residual, sin benchmark contra otras familias y sin usar información futura dentro de cada fold.

**Intervalos al 95%**

Los intervalos salen del mismo bloque residual mediante regresión cuantílica y se recalibran con una capa conformal temporal sobre la muestra de entrenamiento de cada fold. Así evitamos intervalos ingenuos y mantenemos el proceso libre de fuga temporal.

In [ ]:
# Configuracion general del entorno, rutas del repo y estilo grafico del notebook.
import os
import warnings
from pathlib import Path

# Limitamos el paralelismo para evitar problemas de runtime en Windows y mejorar reproducibilidad.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = lambda value: f"{value:,.3f}"

# Detectamos la raiz del repo tanto si se ejecuta desde la carpeta notebooks como desde la raiz.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

RAW_DATA_PATH = REPO_ROOT / "data" / "raw" / "demanda_diaria_metro_2015_2024.csv"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
import os
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

warnings.filterwarnings("ignore")

from dateutil.easter import easter

import holidays
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge


# Semilla unica para que el componente de boosting sea reproducible.
SEED = 42


# Cargamos el CSV bruto, completamos el calendario diario y marcamos fechas imputadas.
def load_metro_series(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["Fecha"]).rename(
        columns={"Fecha": "date", "Demanda": "y"}
    )
    df = df.sort_values("date").drop_duplicates("date")
    # Forzamos una rejilla diaria completa para detectar huecos del fichero original.
    full_idx = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    df = df.set_index("date").reindex(full_idx)
    df.index.name = "date"
    # Conservamos una marca de imputacion para poder reducir el peso de esas fechas al entrenar.
    df["is_imputed"] = df["y"].isna().astype(int)
    df["y"] = df["y"].interpolate(method="time")
    return df.reset_index()


# Construimos todas las variables de calendario que conocemos ex ante para cada fecha.
def build_calendar_frame(dates: pd.DatetimeIndex, fourier_order: int = 6) -> pd.DataFrame:
    idx = pd.DatetimeIndex(pd.to_datetime(dates))
    cal = pd.DataFrame(index=idx)
    # Variables de calendario basicas: posicion semanal, mensual y cercania a inicio/fin de mes.
    cal["dow"] = idx.dayofweek
    cal["is_weekend"] = (cal["dow"] >= 5).astype(int)
    cal["day_of_month"] = idx.day
    cal["week_of_year"] = idx.isocalendar().week.astype(int).values
    cal["month"] = idx.month
    cal["is_month_start"] = idx.is_month_start.astype(int)
    cal["is_month_end"] = idx.is_month_end.astype(int)
    cal["days_to_month_end"] = idx.days_in_month - idx.day
    cal["near_month_end"] = (cal["days_to_month_end"] <= 3).astype(int)
    cal["near_month_start"] = (cal["day_of_month"] <= 3).astype(int)

    # Generamos festivos nacionales y de la Comunidad de Madrid con fechas reales por anio.
    years = sorted(set(idx.year.tolist()) | {idx.min().year - 1, idx.max().year + 1})
    es_national = holidays.country_holidays("ES", years=years)
    es_madrid = holidays.country_holidays("ES", subdiv="MD", years=years)
    national_dates = pd.DatetimeIndex(pd.to_datetime(list(es_national.keys()))).normalize()
    madrid_dates = pd.DatetimeIndex(pd.to_datetime(list(es_madrid.keys()))).normalize()
    madrid_only = madrid_dates.difference(national_dates)
    normalized = idx.normalize()

    cal["is_holiday_national"] = normalized.isin(national_dates).astype(int)
    cal["is_holiday_madrid"] = normalized.isin(madrid_only).astype(int)
    cal["is_holiday_any"] = (
        (cal["is_holiday_national"] + cal["is_holiday_madrid"]) > 0
    ).astype(int)
    cal["is_business_day"] = (
        (cal["is_weekend"] == 0) & (cal["is_holiday_any"] == 0)
    ).astype(int)

    # Definimos dias no laborables para poder crear la variable de puente de forma general.
    non_working = ((cal["is_weekend"] == 1) | (cal["is_holiday_any"] == 1)).astype(int)
    prev_non = pd.Series(non_working, index=idx).shift(1, fill_value=1)
    next_non = pd.Series(non_working, index=idx).shift(-1, fill_value=1)
    holiday_any = pd.Series(cal["is_holiday_any"].values, index=idx)

    cal["is_bridge_day"] = (
        (cal["is_business_day"] == 1) & (prev_non == 1) & (next_non == 1)
    ).astype(int)
    cal["is_holiday_eve"] = (
        (cal["is_business_day"] == 1) & (holiday_any.shift(-1, fill_value=0) == 1)
    ).astype(int)
    cal["is_post_holiday"] = (
        (cal["is_business_day"] == 1) & (holiday_any.shift(1, fill_value=0) == 1)
    ).astype(int)

    # Semana Santa cambia de fecha cada anio, asi que la calculamos a partir de Pascua.
    easter_dates = {year: pd.Timestamp(easter(year)) for year in years}
    holy_week_window = []
    holy_thursday = []
    good_friday = []
    easter_monday = []
    pre_holy_week = []
    post_holy_week = []
    for date in idx.normalize():
        easter_sunday = easter_dates[date.year]
        palm_sunday = easter_sunday - pd.Timedelta(days=7)
        easter_mon = easter_sunday + pd.Timedelta(days=1)
        holy_week_window.append(int(palm_sunday <= date <= easter_mon))
        holy_thursday.append(int(date == easter_sunday - pd.Timedelta(days=3)))
        good_friday.append(int(date == easter_sunday - pd.Timedelta(days=2)))
        easter_monday.append(int(date == easter_mon))
        pre_holy_week.append(int((palm_sunday - pd.Timedelta(days=3)) <= date < palm_sunday))
        post_holy_week.append(int(easter_mon < date <= easter_mon + pd.Timedelta(days=4)))

    cal["is_semana_santa_window"] = holy_week_window
    cal["is_holy_thursday"] = holy_thursday
    cal["is_good_friday"] = good_friday
    cal["is_easter_monday"] = easter_monday
    cal["is_pre_semana_santa"] = pre_holy_week
    cal["is_post_semana_santa"] = post_holy_week

    # Anadimos tendencias e intervenciones para no mezclar COVID con la estructura normal de demanda.
    cal["trend"] = np.arange(len(cal)) / 365.25
    cal["trend_sq"] = cal["trend"] ** 2
    cal["covid_lockdown"] = ((idx >= "2020-03-14") & (idx <= "2020-06-21")).astype(int)
    cal["covid_restrictions"] = ((idx >= "2020-06-22") & (idx <= "2021-06-30")).astype(int)
    cal["post_covid"] = (idx >= "2021-07-01").astype(int)
    cal["trend_post_covid"] = np.where(
        idx >= "2021-07-01",
        (idx - pd.Timestamp("2021-07-01")).days / 365.25,
        0.0,
    )
    cal["trend_post_2023"] = np.where(
        idx >= "2023-01-01",
        (idx - pd.Timestamp("2023-01-01")).days / 365.25,
        0.0,
    )

    # Los terminos de Fourier capturan una estacionalidad anual suave e interpretable.
    day_of_year = idx.dayofyear.values
    for order in range(1, fourier_order + 1):
        cal[f"fourier_sin_{order}"] = np.sin(2 * np.pi * order * day_of_year / 365.25)
        cal[f"fourier_cos_{order}"] = np.cos(2 * np.pi * order * day_of_year / 365.25)

    cal = pd.concat(
        [
            cal,
            pd.get_dummies(cal["dow"], prefix="dow", dtype=float),
            pd.get_dummies(cal["month"], prefix="month", dtype=float),
        ],
        axis=1,
    )

    dow_columns = [col for col in cal.columns if col.startswith("dow_") and "_x_" not in col]
    for dow_col in dow_columns:
        cal[f"{dow_col}_x_month_end"] = cal[dow_col] * cal["near_month_end"]
        cal[f"{dow_col}_x_month_start"] = cal[dow_col] * cal["near_month_start"]
        cal[f"{dow_col}_x_holiday_eve"] = cal[dow_col] * cal["is_holiday_eve"]
        cal[f"{dow_col}_x_post_holiday"] = cal[dow_col] * cal["is_post_holiday"]

    return cal


# Agrupamos variables para explicar despues el forecast en bloques legibles.
def deterministic_feature_groups(columns: List[str]) -> Dict[str, List[str]]:
    groups = {
        "Nivel base": [],
        "Calendario semanal": [],
        "Calendario mensual": [],
        "Festivos y moviles": [],
        "Estacionalidad anual": [],
        "Regimen y tendencia": [],
    }
    for col in columns:
        if col.startswith("fourier_"):
            groups["Estacionalidad anual"].append(col)
        elif (
            "holiday" in col
            or "bridge" in col
            or "semana_santa" in col
            or "holy" in col
            or "easter" in col
        ):
            groups["Festivos y moviles"].append(col)
        elif col.startswith("dow_") or col in {"is_weekend", "is_business_day"}:
            groups["Calendario semanal"].append(col)
        elif col.startswith("month_") or col in {
            "day_of_month",
            "week_of_year",
            "month",
            "is_month_start",
            "is_month_end",
            "days_to_month_end",
            "near_month_end",
            "near_month_start",
        }:
            groups["Calendario mensual"].append(col)
        elif col.startswith("covid") or col.startswith("trend") or col == "post_covid":
            groups["Regimen y tendencia"].append(col)
        else:
            groups["Calendario mensual"].append(col)
    return groups


# Damos mas peso a lo reciente y menos al tramo COVID y a fechas imputadas.
def sample_weights(index: pd.DatetimeIndex, is_imputed: Optional[pd.Series] = None) -> np.ndarray:
    idx = pd.DatetimeIndex(index)
    age_days = np.asarray((idx.max() - idx).days, dtype=float)
    weights = np.power(0.5, age_days / 730.0).astype(float)
    weights[(idx >= "2020-03-14") & (idx <= "2020-06-21")] *= 0.15
    weights[(idx >= "2020-06-22") & (idx <= "2021-06-30")] *= 0.35
    weights[(idx >= "2021-07-01") & (idx < "2023-01-01")] *= 0.75
    if is_imputed is not None:
        weights *= np.where(pd.Series(is_imputed).astype(bool).values, 0.5, 1.0)
    return weights


@dataclass
# La etapa 1 aprende un modelo estructural de calendario y regimen en escala logaritmica.
class DeterministicStage:
    alpha: float = 3.0
    fourier_order: int = 6
    model: Optional[Ridge] = None
    feature_columns: Optional[List[str]] = None
    feature_groups: Optional[Dict[str, List[str]]] = None
    calendar_frame_: Optional[pd.DataFrame] = None

    # Ajustamos el bloque determinista usando solo train, pero dejando preparadas tambien las fechas futuras.
    def fit(self, train_df: pd.DataFrame, forecast_index: pd.DatetimeIndex) -> "DeterministicStage":
        all_index = train_df.index.union(pd.DatetimeIndex(forecast_index))
        calendar = build_calendar_frame(all_index, fourier_order=self.fourier_order)
        feature_columns = [
            "is_weekend",
            "is_business_day",
            "day_of_month",
            "week_of_year",
            "is_month_start",
            "is_month_end",
            "days_to_month_end",
            "near_month_end",
            "near_month_start",
            "is_holiday_national",
            "is_holiday_madrid",
            "is_holiday_eve",
            "is_post_holiday",
            "is_bridge_day",
            "is_semana_santa_window",
            "is_holy_thursday",
            "is_good_friday",
            "is_easter_monday",
            "is_pre_semana_santa",
            "is_post_semana_santa",
            "trend",
            "trend_sq",
            "covid_lockdown",
            "covid_restrictions",
            "post_covid",
            "trend_post_covid",
            "trend_post_2023",
        ]
        feature_columns += [
            col
            for col in calendar.columns
            if col.startswith("fourier_") or col.startswith("dow_") or col.startswith("month_")
        ]
        # Modelar en log1p estabiliza varianza y reduce sensibilidad a valores muy altos.
        y_log = np.log1p(train_df["y"])
        weights = sample_weights(train_df.index, train_df["is_imputed"])
        model = Ridge(alpha=self.alpha)
        model.fit(calendar.loc[train_df.index, feature_columns], y_log, sample_weight=weights)

        self.model = model
        self.feature_columns = feature_columns
        self.feature_groups = deterministic_feature_groups(feature_columns)
        self.calendar_frame_ = calendar
        return self

    def predict_log(self, dates: pd.DatetimeIndex) -> pd.Series:
        return pd.Series(
            self.model.predict(self.calendar_frame_.loc[dates, self.feature_columns]),
            index=pd.DatetimeIndex(dates),
            name="deterministic_log",
        )

    def grouped_contributions(self, dates: pd.DatetimeIndex) -> pd.DataFrame:
        X = self.calendar_frame_.loc[dates, self.feature_columns]
        coef = pd.Series(self.model.coef_, index=self.feature_columns)
        contrib = X.mul(coef, axis=1)
        grouped = pd.DataFrame(index=X.index)
        grouped["Nivel base"] = self.model.intercept_
        for group, cols in self.feature_groups.items():
            if group == "Nivel base":
                continue
            grouped[group] = contrib[cols].sum(axis=1) if cols else 0.0
        grouped["deterministic_log"] = grouped.sum(axis=1)
        return grouped


# Generamos lags y rolling features causales para el residuo y para la serie en log.
def make_origin_features(residual: pd.Series, y_log: pd.Series) -> pd.DataFrame:
    features = pd.DataFrame(index=residual.index)
    features["resid_lag_1"] = residual
    for lag in [2, 3, 7, 14, 21, 28]:
        features[f"resid_lag_{lag}"] = residual.shift(lag - 1)
    for window in [7, 14, 28]:
        features[f"resid_roll_mean_{window}"] = residual.rolling(window).mean()
        features[f"resid_roll_std_{window}"] = residual.rolling(window).std()
        features[f"y_roll_mean_{window}"] = y_log.rolling(window).mean()
        features[f"y_roll_std_{window}"] = y_log.rolling(window).std()
    features["y_lag_1"] = y_log
    features["y_lag_7"] = y_log.shift(6)
    features["y_lag_14"] = y_log.shift(13)
    features["delta_resid_vs_last_week"] = residual - residual.shift(7)
    features["delta_y_vs_last_week"] = y_log - y_log.shift(7)
    return features


# Transformamos la serie residual en un panel supervisado origen-horizonte para forecast directo.
def build_direct_residual_panel(
    residual: pd.Series,
    y_log: pd.Series,
    deterministic_log: pd.Series,
    calendar_frame: pd.DataFrame,
    horizon: int,
    max_training_days: int,
) -> pd.DataFrame:
    origin_features = make_origin_features(residual, y_log)
    valid_origins = origin_features.dropna().index
    if max_training_days is not None and len(valid_origins):
        cutoff = valid_origins.max() - pd.Timedelta(days=max_training_days)
        valid_origins = valid_origins[valid_origins >= cutoff]

    future_calendar_cols = [
        "dow",
        "is_weekend",
        "is_business_day",
        "is_holiday_any",
        "is_bridge_day",
        "is_holiday_eve",
        "is_post_holiday",
        "is_semana_santa_window",
        "is_holy_thursday",
        "is_good_friday",
        "month",
        "near_month_end",
        "near_month_start",
    ]

    rows = []
    # Cada paso h crea ejemplos donde la target es el residuo observado h dias despues del origen.
    for step in range(1, horizon + 1):
        target = residual.shift(-step)
        usable = valid_origins.intersection(target.dropna().index)
        if len(usable) == 0:
            continue
        target_dates = usable + pd.Timedelta(days=step)
        block = origin_features.loc[usable].copy().reset_index(drop=True)
        block["origin_date"] = usable.values
        block["target_date"] = target_dates.values
        block["horizon"] = step
        block["det_target_log"] = deterministic_log.loc[target_dates].values
        block = pd.concat(
            [
                block,
                calendar_frame.loc[target_dates, future_calendar_cols].reset_index(drop=True),
            ],
            axis=1,
        )
        block["target"] = target.loc[usable].values
        rows.append(block)

    panel = pd.concat(rows, ignore_index=True)
    panel["origin_date"] = pd.to_datetime(panel["origin_date"])
    panel["target_date"] = pd.to_datetime(panel["target_date"])
    panel = pd.get_dummies(
        panel,
        columns=["dow", "month"],
        prefix=["target_dow", "target_month"],
        dtype=float,
    )
    return panel


@dataclass
# La etapa 2 aprende solo la parte no explicada por el calendario.
class ResidualStage:
    horizon: int = 30
    max_training_days: int = 730
    calibration_days: int = 180
    alpha: float = 0.05
    model_params: Optional[Dict[str, float]] = None
    point_model: Optional[HistGradientBoostingRegressor] = None
    lower_model: Optional[HistGradientBoostingRegressor] = None
    upper_model: Optional[HistGradientBoostingRegressor] = None
    feature_columns_: Optional[List[str]] = None
    conformal_by_horizon_: Optional[Dict[int, float]] = None

    # Fijamos hiperparametros moderados para equilibrar capacidad predictiva y tiempo de ejecucion.
    def __post_init__(self) -> None:
        if self.model_params is None:
            self.model_params = {
                "learning_rate": 0.05,
                "max_iter": 180,
                "max_depth": 4,
                "min_samples_leaf": 60,
                "l2_regularization": 0.10,
                "early_stopping": False,
                "random_state": SEED,
            }

    def fit(
        self,
        train_df: pd.DataFrame,
        deterministic_log: pd.Series,
        calendar_frame: pd.DataFrame,
    ) -> Tuple[pd.Series, pd.DataFrame]:
        y_log = np.log1p(train_df["y"])
        # El target del segundo bloque es el residuo que deja la etapa determinista.
        residual = pd.Series(y_log - deterministic_log.loc[train_df.index], index=train_df.index)
        panel = build_direct_residual_panel(
            residual=residual,
            y_log=y_log,
            deterministic_log=deterministic_log,
            calendar_frame=calendar_frame,
            horizon=self.horizon,
            max_training_days=self.max_training_days,
        )
        feature_columns = [col for col in panel.columns if col not in {"origin_date", "target_date", "target"}]
        X = panel[feature_columns]
        y = panel["target"]
        weights = sample_weights(panel["origin_date"])

        # Reservamos la cola mas reciente del train para calibrar intervalos sin mirar el futuro.
        latest_origin = panel["origin_date"].max()
        calibration_cutoff = latest_origin - pd.Timedelta(days=self.calibration_days)
        calibration_mask = panel["origin_date"] >= calibration_cutoff
        if calibration_mask.sum() < self.horizon * 30:
            unique_origins = np.sort(panel["origin_date"].unique())
            split_at = max(int(len(unique_origins) * 0.8), 1)
            calibration_origins = set(unique_origins[split_at:])
            calibration_mask = panel["origin_date"].isin(calibration_origins)
        train_mask = ~calibration_mask
        if train_mask.sum() == 0:
            train_mask[:] = True
            calibration_mask[:] = False

        # Ajustamos un modelo puntual y dos cuantiles sobre la misma representacion temporal.
        self.point_model = HistGradientBoostingRegressor(loss="squared_error", **self.model_params)
        self.lower_model = HistGradientBoostingRegressor(
            loss="quantile",
            quantile=0.025,
            **self.model_params,
        )
        self.upper_model = HistGradientBoostingRegressor(
            loss="quantile",
            quantile=0.975,
            **self.model_params,
        )
        self.point_model.fit(X, y, sample_weight=weights)
        self.lower_model.fit(X.loc[train_mask], y.loc[train_mask], sample_weight=weights[train_mask])
        self.upper_model.fit(X.loc[train_mask], y.loc[train_mask], sample_weight=weights[train_mask])
        self.feature_columns_ = feature_columns
        self.conformal_by_horizon_ = {h: 0.0 for h in range(1, self.horizon + 1)}

        # La capa conformal corrige posibles problemas de calibracion de los cuantiles puros.
        if calibration_mask.sum() > 0:
            cal_X = X.loc[calibration_mask]
            cal_y = y.loc[calibration_mask]
            cal_h = panel.loc[calibration_mask, "horizon"]
            lower_pred = self.lower_model.predict(cal_X)
            upper_pred = self.upper_model.predict(cal_X)
            scores = np.maximum(lower_pred - cal_y, np.maximum(cal_y - upper_pred, 0.0))

            def quantile_score(values: np.ndarray) -> float:
                if len(values) == 0:
                    return 0.0
                level = min(1.0, np.ceil((len(values) + 1) * (1 - self.alpha)) / len(values))
                return float(np.quantile(values, level, method="higher"))

            global_adjustment = quantile_score(scores)
            for horizon in range(1, self.horizon + 1):
                horizon_scores = scores[cal_h.values == horizon]
                self.conformal_by_horizon_[horizon] = (
                    quantile_score(horizon_scores) if len(horizon_scores) else global_adjustment
                )
        return residual, panel

    def predict(
        self,
        train_df: pd.DataFrame,
        deterministic_log: pd.Series,
        calendar_frame: pd.DataFrame,
        forecast_index: pd.DatetimeIndex,
    ) -> pd.DataFrame:
        y_log = np.log1p(train_df["y"])
        residual = pd.Series(y_log - deterministic_log.loc[train_df.index], index=train_df.index)
        # En prediccion futura solo podemos usar informacion disponible en el ultimo origen observado.
        latest_origin_features = make_origin_features(residual, y_log).loc[[train_df.index.max()]]
        future_rows = []
        future_calendar_cols = [
            "dow",
            "is_weekend",
            "is_business_day",
            "is_holiday_any",
            "is_bridge_day",
            "is_holiday_eve",
            "is_post_holiday",
            "is_semana_santa_window",
            "is_holy_thursday",
            "is_good_friday",
            "month",
            "near_month_end",
            "near_month_start",
        ]
        for step, target_date in enumerate(pd.DatetimeIndex(forecast_index), start=1):
            row = latest_origin_features.copy().reset_index(drop=True)
            row["horizon"] = step
            row["det_target_log"] = deterministic_log.loc[target_date]
            row = pd.concat(
                [
                    row,
                    calendar_frame.loc[[target_date], future_calendar_cols].reset_index(drop=True),
                ],
                axis=1,
            )
            future_rows.append(row)

        future_X = pd.concat(future_rows, ignore_index=True)
        future_X = pd.get_dummies(
            future_X,
            columns=["dow", "month"],
            prefix=["target_dow", "target_month"],
            dtype=float,
        )
        # Reindexamos columnas para replicar exactamente el espacio de entrenamiento.
        future_X = future_X.reindex(columns=self.feature_columns_, fill_value=0.0)

        residual_point = self.point_model.predict(future_X)
        residual_lower = self.lower_model.predict(future_X)
        residual_upper = self.upper_model.predict(future_X)
        # Ensanchamos los cuantiles segun el ajuste conformal especifico de cada horizonte.
        adjustments = np.array(
            [self.conformal_by_horizon_.get(h, 0.0) for h in range(1, len(forecast_index) + 1)],
            dtype=float,
        )
        residual_lower = residual_lower - adjustments
        residual_upper = residual_upper + adjustments

        return pd.DataFrame(
            {
                "date": pd.DatetimeIndex(forecast_index),
                "residual_log": residual_point,
                "residual_lower_log": residual_lower,
                "residual_upper_log": residual_upper,
            }
        )


@dataclass
# Esta clase une ambas etapas y expone una interfaz simple de entrenamiento y prediccion.
class MetroHybridForecaster:
    horizon: int = 30
    deterministic_alpha: float = 3.0
    fourier_order: int = 6
    residual_history_days: int = 730
    deterministic_stage_: Optional[DeterministicStage] = None
    residual_stage_: Optional[ResidualStage] = None

    # Primero aprendemos la parte explicable; despues modelamos el residuo que queda.
    def fit(self, train_df: pd.DataFrame, forecast_index: pd.DatetimeIndex) -> "MetroHybridForecaster":
        deterministic_stage = DeterministicStage(
            alpha=self.deterministic_alpha,
            fourier_order=self.fourier_order,
        ).fit(train_df, forecast_index)
        residual_stage = ResidualStage(
            horizon=self.horizon,
            max_training_days=self.residual_history_days,
        )
        deterministic_log = deterministic_stage.predict_log(deterministic_stage.calendar_frame_.index)
        residual_stage.fit(
            train_df=train_df,
            deterministic_log=deterministic_log,
            calendar_frame=deterministic_stage.calendar_frame_,
        )

        self.deterministic_stage_ = deterministic_stage
        self.residual_stage_ = residual_stage
        return self

    # Sumamos ambas etapas y devolvemos forecast puntual, intervalos y descomposicion interpretable.
    def predict(self, train_df: pd.DataFrame, forecast_index: pd.DatetimeIndex) -> pd.DataFrame:
        deterministic_log = self.deterministic_stage_.predict_log(forecast_index)
        grouped = self.deterministic_stage_.grouped_contributions(forecast_index)
        residual_forecast = self.residual_stage_.predict(
            train_df=train_df,
            deterministic_log=self.deterministic_stage_.predict_log(
                self.deterministic_stage_.calendar_frame_.index
            ),
            calendar_frame=self.deterministic_stage_.calendar_frame_,
            forecast_index=forecast_index,
        ).set_index("date")

        forecast = pd.DataFrame(index=pd.DatetimeIndex(forecast_index))
        forecast["deterministic_log"] = deterministic_log
        forecast["residual_log"] = residual_forecast["residual_log"]
        forecast["yhat"] = np.clip(
            np.expm1(forecast["deterministic_log"] + forecast["residual_log"]),
            a_min=0,
            a_max=None,
        )
        forecast["lower_95"] = np.clip(
            np.expm1(forecast["deterministic_log"] + residual_forecast["residual_lower_log"]),
            a_min=0,
            a_max=None,
        )
        forecast["upper_95"] = np.clip(
            np.expm1(forecast["deterministic_log"] + residual_forecast["residual_upper_log"]),
            a_min=0,
            a_max=None,
        )
        forecast["deterministic_part"] = np.clip(
            np.expm1(forecast["deterministic_log"]),
            a_min=0,
            a_max=None,
        )
        forecast["residual_part"] = forecast["yhat"] - forecast["deterministic_part"]
        forecast = forecast.join(grouped.drop(columns="deterministic_log"))
        forecast.index.name = "date"
        return forecast.reset_index()


# Backtesting rolling-origin: reentrenamos el pipeline completo para cada origen de 2024.
def run_backtest(
    series_df: pd.DataFrame,
    start: str = "2024-01-01",
    step_days: int = 7,
    horizon: int = 30,
) -> pd.DataFrame:
    series_df = series_df.copy().set_index("date")
    last_origin = series_df.index.max() - pd.Timedelta(days=horizon)
    origins = pd.date_range(start, last_origin, freq=f"{step_days}D")
    forecasts = []
    # Cada fold usa todo el historico hasta origin y predice los 30 dias siguientes.
    for origin in origins:
        train_df = series_df.loc[:origin]
        forecast_index = pd.date_range(origin + pd.Timedelta(days=1), periods=horizon, freq="D")
        model = MetroHybridForecaster(horizon=horizon)
        model.fit(train_df, forecast_index)
        fold_forecast = model.predict(train_df, forecast_index)
        fold_forecast["origin_date"] = origin
        fold_forecast["horizon"] = np.arange(1, horizon + 1)
        fold_forecast["actual"] = series_df.loc[forecast_index, "y"].values
        forecasts.append(fold_forecast)
    return pd.concat(forecasts, ignore_index=True)


# sMAPE aporta una medida porcentual mas estable que MAPE cuando hay distintos niveles.
def smape(y_true: pd.Series, y_pred: pd.Series) -> float:
    denom = np.abs(y_true) + np.abs(y_pred)
    ratio = np.where(denom == 0, 0.0, 2 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(ratio))


# Resumimos el backtest tanto a nivel global como por horizonte para diagnostico fino.
def summarize_backtest(backtest_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    bt = backtest_df.copy()
    bt["error"] = bt["yhat"] - bt["actual"]
    bt["abs_error"] = bt["error"].abs()
    bt["sq_error"] = bt["error"] ** 2
    bt["inside_ic95"] = ((bt["actual"] >= bt["lower_95"]) & (bt["actual"] <= bt["upper_95"])).astype(int)
    bt["interval_width"] = bt["upper_95"] - bt["lower_95"]

    global_summary = pd.DataFrame(
        {
            "Metrica": [
                "MAE global",
                "RMSE global",
                "sMAPE global",
                "Sesgo medio",
                "Cobertura IC95",
                "Anchura media IC95",
                "% fuera IC95",
            ],
            "Valor": [
                bt["abs_error"].mean(),
                np.sqrt(bt["sq_error"].mean()),
                smape(bt["actual"], bt["yhat"]),
                bt["error"].mean(),
                bt["inside_ic95"].mean(),
                bt["interval_width"].mean(),
                1 - bt["inside_ic95"].mean(),
            ],
        }
    )

    horizon_summary = (
        bt.groupby("horizon")
        .agg(
            mae=("abs_error", "mean"),
            rmse=("sq_error", lambda s: np.sqrt(np.mean(s))),
            mean_error=("error", "mean"),
            coverage_95=("inside_ic95", "mean"),
            interval_width=("interval_width", "mean"),
        )
        .reset_index()
    )
    return global_summary, horizon_summary

In [ ]:
# Funciones auxiliares de visualizacion para diagnosticar la serie y el backtesting.
# Estas funciones producen las figuras principales del notebook.
def plot_series_overview(series_df: pd.DataFrame) -> None:
    # Primero mostramos la serie completa y despues un zoom en el tramo reciente con COVID.
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)

    axes[0].plot(series_df["date"], series_df["y"], color="tab:blue", linewidth=1.1)
    axes[0].set_title("Demanda diaria de Metro Madrid (serie completa)")
    axes[0].set_ylabel("Viajeros")

    recent = series_df.query("date >= '2019-01-01'")
    axes[1].plot(recent["date"], recent["y"], color="tab:orange", linewidth=1.2, label="Demanda")
    axes[1].axvspan(pd.Timestamp("2020-03-14"), pd.Timestamp("2020-06-21"), color="tab:red", alpha=0.12, label="Lockdown")
    axes[1].axvspan(pd.Timestamp("2020-06-22"), pd.Timestamp("2021-06-30"), color="tab:purple", alpha=0.08, label="Restricciones")
    axes[1].set_title("Zoom 2019-2024: ruptura COVID y recuperación")
    axes[1].set_ylabel("Viajeros")
    axes[1].legend(loc="upper left")

    plt.tight_layout()
    plt.show()


# Esta figura comprueba si la etapa 1 absorbe bien la parte explicable de la serie.
def plot_deterministic_diagnostics(series_df: pd.DataFrame, model: MetroHybridForecaster) -> None:
    fitted = series_df.set_index("date").copy()
    det_log = model.deterministic_stage_.predict_log(fitted.index)
    fitted["deterministic_part"] = np.clip(np.expm1(det_log), a_min=0, a_max=None)
    fitted["residual_log"] = np.log1p(fitted["y"]) - det_log

    # Primero mostramos la serie completa y despues un zoom en el tramo reciente con COVID.
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)
    tail = fitted.loc["2023-01-01":].reset_index()
    axes[0].plot(tail["date"], tail["y"], label="Real", color="black", linewidth=1.2)
    axes[0].plot(tail["date"], tail["deterministic_part"], label="Parte determinista", color="tab:blue", linewidth=1.4)
    axes[0].set_title("Ajuste de la etapa 1 sobre el tramo reciente")
    axes[0].set_ylabel("Viajeros")
    axes[0].legend(loc="upper left")

    axes[1].plot(fitted.index, fitted["residual_log"], color="tab:green", linewidth=0.8)
    axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1.0)
    axes[1].set_title("Residuo en escala log tras retirar la parte determinista")
    axes[1].set_ylabel("Residuo log")

    plt.tight_layout()
    plt.show()

    print(
        "Std log(y):",
        round(float(np.log1p(series_df['y']).std()), 4),
        "| Std residuo log:",
        round(float(fitted["residual_log"].std()), 4),
    )


# Elegimos unos pocos folds para no saturar el notebook con 44 paneles casi iguales.
def plot_selected_folds(backtest_df: pd.DataFrame, n_folds: int = 4) -> None:
    origins = pd.Series(backtest_df["origin_date"].drop_duplicates().sort_values().tolist())
    if len(origins) <= n_folds:
        selected = origins.tolist()
    else:
        positions = np.linspace(0, len(origins) - 1, n_folds).round().astype(int)
        selected = origins.iloc[positions].tolist()

    fig, axes = plt.subplots(len(selected), 1, figsize=(16, 4 * len(selected)), sharex=False)
    if len(selected) == 1:
        axes = [axes]

    for axis, origin in zip(axes, selected):
        fold = backtest_df.loc[backtest_df["origin_date"] == origin].copy()
        axis.plot(fold["date"], fold["actual"], color="black", linewidth=1.3, label="Real")
        axis.plot(fold["date"], fold["yhat"], color="tab:blue", linewidth=1.3, label="Predicción")
        axis.fill_between(fold["date"], fold["lower_95"], fold["upper_95"], color="tab:blue", alpha=0.15, label="IC95")
        axis.set_title(f"Fold con origen {pd.Timestamp(origin).date()} (horizonte 30 días)")
        axis.set_ylabel("Viajeros")
        axis.legend(loc="upper left")

    plt.tight_layout()
    plt.show()


# El comportamiento por horizonte ayuda a ver donde se degrada antes el modelo.
def plot_horizon_diagnostics(horizon_summary: pd.DataFrame) -> None:
    fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

    axes[0].plot(horizon_summary["horizon"], horizon_summary["mae"], color="tab:red", linewidth=2)
    axes[0].set_title("MAE por horizonte")
    axes[0].set_ylabel("MAE")

    axes[1].plot(horizon_summary["horizon"], horizon_summary["coverage_95"], color="tab:green", linewidth=2)
    axes[1].axhline(0.95, color="black", linestyle="--", linewidth=1, label="Objetivo 95%")
    axes[1].set_title("Cobertura empírica del IC95 por horizonte")
    axes[1].set_ylabel("Cobertura")
    axes[1].legend(loc="lower right")

    axes[2].plot(horizon_summary["horizon"], horizon_summary["interval_width"], color="tab:purple", linewidth=2)
    axes[2].set_title("Anchura media del intervalo por horizonte")
    axes[2].set_ylabel("Anchura media")
    axes[2].set_xlabel("Horizonte (días)")

    plt.tight_layout()
    plt.show()

## 1. Carga y validación de la serie

Partimos del CSV bruto del repo y lo convertimos a frecuencia diaria completa. El fichero original tiene un hueco de 31 días en octubre de 2016; aquí lo rellenamos por interpolación temporal y lo marcamos con `is_imputed` para poder ponderarlo menos en el entrenamiento. Ese hueco queda muy lejos del periodo de backtesting 2024, así que no contamina la evaluación final.

In [ ]:
# Cargamos la serie limpia y comprobamos que su rango e imputaciones sean los esperados.
series = load_metro_series(str(RAW_DATA_PATH))
display(series.head())

expected_days = (series["date"].max() - series["date"].min()).days + 1
print("Rango:", series["date"].min().date(), "a", series["date"].max().date())
print("Observaciones:", len(series), "| Días esperados:", expected_days)
print("Fechas imputadas por huecos del dataset original:", int(series["is_imputed"].sum()))
print("Última fecha disponible para entrenar el forecast final:", series["date"].max().date())

In [ ]:
# Revisamos la forma general de la serie antes de modelar.
plot_series_overview(series)

imputed_window = series.loc[series["is_imputed"].eq(1), ["date", "y"]]
if not imputed_window.empty:
    print("Primeras fechas imputadas del hueco heredado del CSV original:")
    display(imputed_window.head(10))

## 2. Diagnóstico rápido de la etapa determinista

Antes del backtesting conviene ver si la primera etapa está haciendo el trabajo correcto: explicar la estructura repetitiva y dejar un residuo más compacto. Este bloque no sustituye al backtest; solo sirve para comprobar que la separación entre parte explicable y no explicable tiene sentido.

In [ ]:
# Ajustamos una primera version del pipeline para inspeccionar solo la etapa determinista.
series_idx = series.set_index("date")
future_stub = pd.date_range(series["date"].max() + pd.Timedelta(days=1), periods=30, freq="D")
demo_model = MetroHybridForecaster(horizon=30)
demo_model.fit(series_idx, future_stub)
plot_deterministic_diagnostics(series, demo_model)

future_grouped = demo_model.deterministic_stage_.grouped_contributions(future_stub)
print("Contribuciones agrupadas de la etapa determinista para el próximo mes:")
display(future_grouped.head(10))

## 3. Backtesting rolling-origin sobre 2024

Configuración elegida:
- periodo de evaluación: todo 2024 con ventanas de test de 30 días,
- origen móvil con `step=7` días,
- reentrenamiento completo del pipeline en cada fold,
- reconstrucción de todas las features usando solo información disponible hasta el origen.

El `step` semanal evita 300+ reentrenamientos casi redundantes, pero sigue siendo suficientemente exigente para un caso operativo diario y produce 44 folds en 2024.

In [ ]:
# Ejecutamos el rolling-origin 2024 con step semanal y horizonte fijo de 30 dias.
backtest_results = run_backtest(
    series_df=series,
    start="2024-01-01",
    step_days=7,
    horizon=30,
)

# Normalizamos tipos de fecha para tablas y graficas.
backtest_results["date"] = pd.to_datetime(backtest_results["date"])
backtest_results["origin_date"] = pd.to_datetime(backtest_results["origin_date"])
global_summary, horizon_summary = summarize_backtest(backtest_results)

In [ ]:
# Primero leemos tablas agregadas y despues graficas de diagnostico del rolling-origin 2024.
print("Resumen global del rolling-origin 2024")
display(global_summary)

print("Resumen por horizonte")
display(horizon_summary.head(30))

plot_selected_folds(backtest_results, n_folds=4)
plot_horizon_diagnostics(horizon_summary)

## 4. Forecast final de los próximos 30 días

Una vez validado el pipeline, se reentrena con todo el histórico disponible y se genera el forecast operativo a 30 días. Además del `yhat` y del intervalo al 95%, guardamos la parte determinista, la parte residual prevista y una descomposición agrupada de contribuciones del calendario.

In [ ]:
# Finalmente reentrenamos con todo el historico y generamos el forecast operativo a 30 dias.
final_train = series.set_index("date")
forecast_index = pd.date_range(series["date"].max() + pd.Timedelta(days=1), periods=30, freq="D")

final_model = MetroHybridForecaster(horizon=30)
final_model.fit(final_train, forecast_index)
final_forecast = final_model.predict(final_train, forecast_index)

# Estas son las columnas finales que queremos enseñar y exportar al CSV.
ordered_cols = [
    "date",
    "yhat",
    "lower_95",
    "upper_95",
    "deterministic_part",
    "residual_part",
    "Nivel base",
    "Calendario semanal",
    "Calendario mensual",
    "Festivos y moviles",
    "Estacionalidad anual",
    "Regimen y tendencia",
]
display(final_forecast[ordered_cols].head(30))

output_path = OUTPUT_DIR / "metro_custom_short_term_forecast.csv"
final_forecast[ordered_cols].to_csv(output_path, index=False)
print(f"Forecast guardado en: {output_path}")

## 5. Limitaciones y mejoras futuras

- La transformación `log1p` mejora estabilidad y robustez, pero la vuelta a escala original con `expm1` hace que el punto central se interprete más como mediana condicional que como media exacta.
- Los intervalos están calibrados temporalmente y funcionan bien en cobertura, pero siguen siendo relativamente anchos en algunos horizontes; es el precio de proteger la incertidumbre residual en una serie con shocks de calendario muy marcados.
- El bloque residual solo usa información endógena y calendario. Si en un caso real se dispusiera de huelgas, meteorología, eventos masivos o incidencias de red, probablemente habría margen adicional de mejora.
- La imputación del hueco de octubre de 2016 es razonable por estar lejos del test 2024, pero si se necesitara un modelo histórico más estructural convendría contrastarla con la fuente original.
- Si el objetivo cambiara de corto plazo a medio/largo plazo, habría que replantear tanto la ponderación temporal como la agresividad del bloque residual.